In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_process_stage_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalProcessStageId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "LegalProcessStageKey string,"
        #     "LegalProcessKey string,"
        #     "ExposureKey string,"
        #     "PartyKey string,"
        #     "CDI string,"
        #     "CaseCode string,"
        #     "LegalProcessCreatedBy string,"
        #     "LegalProcessCreatedDate timestamp,"
        #     "LegalProcessStartDate timestamp,"
        #     "LegalProcessEndDate string,"
        #     "LegalProcessDefinitionCode string,"
        #     "LegalProcessDefinition string,"
        #     "LegalProcessStatusCode int,"
        #     "LegalProcessStatus string,"
        #     "ExternalCompanyAssignmentKey int,"
        #     "LegalProcessReferenceNumber string,"
        #     "LegalProcessControlAreaKey int,"
        #     "LegalStageKey string,"
        #     "LegalStageCreatedBy string,"
        #     "LegalStageCreatedDate timestamp,"
        #     "LegalStageStartDate timestamp,"
        #     "LegalStageEndDate timestamp,"
        #     "LegalStageDefinitionCode int,"
        #     "LegalStageDefinition string,"
        #     "LegalStageCurrentFlag boolean,"
        #     "LegalStageFinalFlag string,"
        #     "ParalellLegalStageKey string,"
        #     "ParalellLegalStageCreatedBy string,"
        #     "ParalellLegalStageCreatedDate timestamp,"
        #     "ParalellLegalStageStartDate timestamp,"
        #     "ParalellLegalStageDefinitionCode int,"
        #     "ParalellLegalStageDefinition string,"
        #     "RelatedLegalStageKey int,"
        #     "LegalStagePiteKey int,"
        #     "LegalStagePreviousPiteKey int,"
        #     "LegalStageFspishDuration int,"
        #     "BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalProcessStage"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== case_master ==================
        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "case_code",
            "debt_code",
            "vend_code",
            "old_case_code",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",  # check name
            case_master_final_filters,
            case_master_selected_fields,
        )

        # ================== process_instances ==================

        process_instances_extra_filters = None

        if process_instances_extra_filters is not None:
            process_instances_extra_filters.append(date_filter)
            process_instances_final_filters = " AND ".join(
                process_instances_extra_filters
            )
        else:
            process_instances_final_filters = date_filter

        process_instances_selected_fields = [
            "EXTC_CODE",
            "prin_code",
            "cust_code",
            "PRIN_STATUS",
            "PRIN_CREATED_BY",
            "PRIN_CREATED",
            "PRIN_START_DATE",
            "PRIN_END_DATE",
            "prdf_code",
            "businessdate",
        ]

        process_instances = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instances",  # check name
            process_instances_final_filters,
            process_instances_selected_fields,
        )
        process_instances = process_instances.withColumnRenamed(
            "prin_code", "prin_code_pi"
        )
        process_instances = process_instances.withColumn(
            "prin_code_pi", col("prin_code_pi").cast("string")
        )
        process_instances = process_instances.withColumn(
            "cust_code", col("cust_code").cast("string")
        )
        process_instances = process_instances.withColumnRenamed(
            "prdf_code", "prdf_code_pi"
        )
        # ================== process_instance_stages ==================

        process_instance_stages_extra_filters = None

        if process_instance_stages_extra_filters is not None:
            process_instance_stages_extra_filters.append(date_filter)
            process_instance_stages_final_filters = " AND ".join(
                process_instance_stages_extra_filters
            )
        else:
            process_instance_stages_final_filters = date_filter

        process_instance_stages_selected_fields = [
            "PRIS_CODE",
            "PRIS_CREATED_BY",
            "PRIS_CREATED",
            "PRIS_STARTED_ON",
            "PRIS_ENDED_ON",
            "PDST_CODE",
            "PRIS_IS_CURRENT_FLAG",
            "PRIN_CODE",
        ]

        process_instance_stages = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instance_stages",  # check name
            process_instance_stages_final_filters,
            process_instance_stages_selected_fields,
        )
        process_instance_stages = process_instance_stages.withColumnRenamed(
            "PRIS_CODE", "pris_code_pis"
        )
        process_instance_stages = process_instance_stages.withColumn(
            "pris_code_pis", col("pris_code_pis").cast("string")
        )
        process_instance_stages = process_instance_stages.withColumnRenamed(
            "PDST_CODE", "pdst_code_pis"
        )
        # ================== process_instance_participants ==================

        process_instance_participants_extra_filters = None

        if process_instance_participants_extra_filters is not None:
            process_instance_participants_extra_filters.append(date_filter)
            process_instance_participants_final_filters = " AND ".join(
                process_instance_participants_extra_filters
            )
        else:
            process_instance_participants_final_filters = date_filter

        process_instance_participants_selected_fields = [
            "CASE_CODE",
            "cust_code",
            "PRIN_CODE",
        ]

        process_instance_participants = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instance_participants",  # check name
            process_instance_participants_final_filters,
            process_instance_participants_selected_fields,
        )
        process_instance_participants = process_instance_participants.withColumnRenamed(
            "CASE_CODE", "case_code_pip"
        )
        process_instance_participants = process_instance_participants.withColumn(
            "cust_code", col("cust_code").cast("string")
        )
        process_instance_participants = process_instance_participants.withColumn(
            "case_code_pip", col("case_code_pip").cast("string")
        )
        # ================== process_definitions ==================

        process_definitions_extra_filters = None

        if process_definitions_extra_filters is not None:
            process_definitions_extra_filters.append(date_filter)
            process_definitions_final_filters = " AND ".join(
                process_definitions_extra_filters
            )
        else:
            process_definitions_final_filters = date_filter

        process_definitions_selected_fields = ["PRDF_CODE", "PRDF_NAME"]

        process_definitions = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_definitions",  # check name
            process_definitions_final_filters,
            process_definitions_selected_fields,
        )
        # ================== process_definition_stages ==================

        process_definition_stages_extra_filters = None

        if process_definition_stages_extra_filters is not None:
            process_definition_stages_extra_filters.append(date_filter)
            process_definition_stages_final_filters = " AND ".join(
                process_definition_stages_extra_filters
            )
        else:
            process_definition_stages_final_filters = date_filter

        process_definition_stages_selected_fields = ["PDST_NAME", "PDST_CODE"]

        process_definition_stages = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_definition_stages",  # check name
            process_definition_stages_final_filters,
            process_definition_stages_selected_fields,
        )
        # ================== process_definition_parallel_stages ==================

        process_definition_parallel_stages_extra_filters = None

        if process_definition_parallel_stages_extra_filters is not None:
            process_definition_parallel_stages_extra_filters.append(date_filter)
            process_definition_parallel_stages_final_filters = " AND ".join(
                process_definition_parallel_stages_extra_filters
            )
        else:
            process_definition_parallel_stages_final_filters = date_filter

        process_definition_parallel_stages_selected_fields = ["PDPS_NAME", "pdps_code"]

        process_definition_parallel_stages = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_definition_parallel_stages",  # check name
            process_definition_parallel_stages_final_filters,
            process_definition_parallel_stages_selected_fields,
        )
        # ================== process_instance_parallel_stages ==================

        process_instance_parallel_stages_extra_filters = None

        if process_instance_parallel_stages_extra_filters is not None:
            process_instance_parallel_stages_extra_filters.append(date_filter)
            process_instance_parallel_stages_final_filters = " AND ".join(
                process_instance_parallel_stages_extra_filters
            )
        else:
            process_instance_parallel_stages_final_filters = date_filter

        process_instance_parallel_stages_selected_fields = [
            "PIPS_CODE",
            "PIPS_CREATED_BY",
            "PIPS_CREATED",
            "PIPS_START_DATE",
            "PDPS_CODE",
            "PRIS_CODE",
            "PRIN_CODE",
        ]

        process_instance_parallel_stages = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_process_instance_parallel_stages",  # check name
            process_instance_parallel_stages_final_filters,
            process_instance_parallel_stages_selected_fields,
        )
        process_instance_parallel_stages = (
            process_instance_parallel_stages.withColumnRenamed(
                "PDPS_CODE", "pdps_code_pips"
            )
        )
        process_instance_parallel_stages = process_instance_parallel_stages.withColumn(
            "PIPS_CODE", col("PIPS_CODE").cast("string")
        )

        # ================== lst_of_val ==================

        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = ["lov_desc", "lovt_code", "LOV_LOW"]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",  # check name
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )

        # ================== portfolio ==================
        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "vendor_code_qualco",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== customer ==================
        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cust_code",
            "cdi",
        ]

        customer = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== Join Conditions==================

        process_instances_join_condition = (
            process_instances.join(
                process_definitions,
                (process_instances["prdf_code_pi"] == process_definitions["prdf_code"]),
                "left",
            )
            .join(
                lst_of_val,
                (process_instances["prin_status"] == lst_of_val["lov_low"]),
                "left",
            )
            .where(lst_of_val["lovt_code"] == "627")
            .join(
                process_instance_participants,
                (
                    process_instances["prin_code_pi"]
                    == process_instance_participants["prin_code"]
                ),
                "left",
            )
            .join(
                process_instance_stages,
                (
                    process_instances["prin_code_pi"]
                    == process_instance_stages["prin_code"]
                ),
                "left",
            )
            .join(
                process_instance_parallel_stages,
                (
                    process_instances["prin_code_pi"]
                    == process_instance_parallel_stages["prin_code"]
                ),
                "left",
            )
            .join(
                customer,
                (
                    customer["cust_code"]
                    == coalesce(
                        process_instance_participants["cust_code"],
                        process_instances["cust_code"],
                    )
                ),
                "left",
            )
        )

        legal_process_stage_qc_join_condition = (
            process_instances_join_condition.join(
                process_definition_parallel_stages,
                (
                    process_instances_join_condition["pdps_code_pips"]
                    == process_definition_parallel_stages["pdps_code"]
                ),
                "left",
            )
            .join(
                case_master,
                (
                    process_instances_join_condition["case_code_pip"]
                    == case_master["case_code"]
                ),
                "left",
            )
            .join(
                portfolio,
                (case_master["vend_code"] == portfolio["vendor_code_qualco"]),
                "left",
            )
            .join(
                process_definition_stages,
                (
                    process_instances_join_condition["pdst_code_pis"]
                    == process_definition_stages["pdst_code"]
                ),
                "left",
            )
        )

        legal_process_stage_qc_primary_key_fields = [
            process_instances.prin_code_pi,
            process_instance_stages.pris_code_pis,
            process_instance_parallel_stages.PIPS_CODE,
            process_instance_participants.cust_code,
            process_instances.cust_code,
            process_instance_participants.case_code_pip,
            lit("qc"),
        ]
        LegalProcessStageKey_field = [
            coalesce(process_instances.prin_code_pi, lit("N/A")),
            coalesce(process_instance_stages.pris_code_pis, lit("N/A")),
            coalesce(process_instance_parallel_stages.PIPS_CODE, lit("N/A")),
            coalesce(process_instance_participants.cust_code, lit("N/A")),
            coalesce(process_instances.cust_code, lit("N/A")),
            coalesce(process_instance_participants.case_code_pip, lit("N/A")),
        ]

        legal_process_stage_qc = (
            legal_process_stage_qc_join_condition.withColumn(
                "LegalProcessStageId",
                sha2(
                    concat_ws("", *legal_process_stage_qc_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "LegalProcessStageKey", concat_ws(";", *LegalProcessStageKey_field)
            )
            .withColumn(
                "PartyKey",
                coalesce(
                    process_instance_participants["cust_code"],
                    process_instances["cust_code"],
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        legal_process_stage_qc = legal_process_stage_qc.select(
            legal_process_stage_qc.LegalProcessStageId.cast("String").alias(
                "LegalProcessStageId"
            ),
            lit("qc").alias("SourceSystem"),
            case_master.vend_code.cast("String").alias("PortfolioKey"),        # int
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            legal_process_stage_qc.LegalProcessStageKey.cast("String").alias(
                "LegalProcessStageKey"
            ),
            legal_process_stage_qc.prin_code_pi.cast("string").alias("LegalProcessKey"),
            case_master.old_case_code.alias("ExposureKey"),
            legal_process_stage_qc.PartyKey.cast("string").alias("PartyKey"),
            legal_process_stage_qc.cdi.cast("string").alias("CDI"),
            legal_process_stage_qc.case_code_pip.cast("string").alias("CaseCode"),
            legal_process_stage_qc.PRIN_CREATED_BY.alias("LegalProcessCreatedBy"),
            legal_process_stage_qc.PRIN_CREATED.alias("LegalProcessCreatedDate"),
            legal_process_stage_qc.PRIN_START_DATE.alias("LegalProcessStartDate"),
            legal_process_stage_qc.PRIN_END_DATE.cast("string").alias(
                "LegalProcessEndDate"
            ),
            legal_process_stage_qc.PRDF_CODE.cast("string").alias(
                "LegalProcessDefinitionCode"
            ),
            legal_process_stage_qc.PRDF_NAME.alias("LegalProcessDefinition"),
            legal_process_stage_qc.PRIN_STATUS.alias("LegalProcessStatusCode"),
            legal_process_stage_qc.lov_desc.alias("LegalProcessStatus"),
            legal_process_stage_qc.EXTC_CODE.alias("ExternalCompanyAssignmentKey"),
            lit(None).alias("LegalProcessReferenceNumber"),
            lit(None).alias("LegalProcessControlAreaKey"),
            legal_process_stage_qc.pris_code_pis.cast("string").alias("LegalStageKey"),
            legal_process_stage_qc.PRIS_CREATED_BY.alias("LegalStageCreatedBy"),
            legal_process_stage_qc.PRIS_CREATED.alias("LegalStageCreatedDate"),
            legal_process_stage_qc.PRIS_STARTED_ON.alias("LegalStageStartDate"),
            legal_process_stage_qc.PRIS_ENDED_ON.alias("LegalStageEndDate"),
            legal_process_stage_qc.pdst_code_pis.alias("LegalStageDefinitionCode"),
            legal_process_stage_qc.PDST_NAME.alias("LegalStageDefinition"),
            legal_process_stage_qc.PRIS_IS_CURRENT_FLAG.alias("LegalStageCurrentFlag"),
            lit(None).alias("LegalStageFinalFlag"),
            legal_process_stage_qc.PIPS_CODE.cast("string").alias(
                "ParalellLegalStageKey"
            ),
            legal_process_stage_qc.PIPS_CREATED_BY.alias("ParalellLegalStageCreatedBy"),
            legal_process_stage_qc.PIPS_CREATED.alias("ParalellLegalStageCreatedDate"),
            legal_process_stage_qc.PIPS_START_DATE.alias("ParalellLegalStageStartDate"),
            legal_process_stage_qc.pdps_code_pips.alias(
                "ParalellLegalStageDefinitionCode"
            ),
            legal_process_stage_qc.PDPS_NAME.alias("ParalellLegalStageDefinition"),
            legal_process_stage_qc.PRIS_CODE.alias("RelatedLegalStageKey"),
            lit(None).alias("LegalStagePiteKey"),
            lit(None).alias("LegalStagePreviousPiteKey"),
            lit(None).alias("LegalStageFspishDuration"),
            legal_process_stage_qc.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        legal_process_stage_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_legal_process_stage_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)